# Decision Tree

In [ ]:
import sys; sys.path.insert(0, "..")

from utils.dependencies import *
from utils import (
    load_and_filter, impute, split_and_balance, FEATURES,
    get_class_meta,
    run_grid_search,
    run_stratified_cv,
    run_seed_sweep, SweepResult,
    print_sweep_classification_report,
    print_sweep_performance_summary,
    print_sweep_error_patterns,
    plot_posterior_violins,
    plot_sweep_f1_bar, plot_sweep_confusion_matrix, plot_sweep_feature_importances,
)

## Model-Specific Data Processing

In [ ]:
df = load_and_filter()
df = impute(df)

# Decision trees split on raw thresholds — log transforms and clipping are not needed.
# StandardScaler is passed for interface consistency but does not affect tree behaviour.
X_train_bal, y_train_bal, X_test_scaled, y_test, class_weight_dict, scaler = split_and_balance(
    df,
    scaler=StandardScaler(),
    smote_variant="standard",
)

class_names, classes, colors = get_class_meta()

## Training

In [ ]:
param_grid = {
    "max_depth": [3, 5, 7, 10, None],
    "min_samples_leaf": [1, 2, 5, 10],
    "criterion": ["gini", "entropy"],
}

grid_dt = run_grid_search(
    DecisionTreeClassifier(class_weight=class_weight_dict, random_state=42),
    param_grid,
    X_train_bal,
    y_train_bal,
)
dt = grid_dt.best_estimator_

y_pred_dt = dt.predict(X_test_scaled)
y_prob_dt  = dt.predict_proba(X_test_scaled)

print("Decision Tree training complete.")
print(f"Best max_depth:        {grid_dt.best_params_['max_depth']}")
print(f"Best min_samples_leaf: {grid_dt.best_params_['min_samples_leaf']}")
print(f"Best criterion:        {grid_dt.best_params_['criterion']}")
print(f"Best CV F1 Macro:      {grid_dt.best_score_:.4f}")
print(f"Actual tree depth:     {dt.get_depth()}")
print(f"Number of leaves:      {dt.get_n_leaves()}")

### Seed-Sweep Evaluation

Grid search above tuned hyperparameters on `random_state=42` only. The single test F1 macro from that one split is a noisy estimate when the minority test classes are tiny (6 mesoplanet, 8 psychroplanet samples). Re-evaluate with the same tuned hyperparameters across 50 random splits to estimate expected performance — and capture per-seed confusion matrices and feature importances for downstream aggregated diagnostics.

In [ ]:
sweep_dt = run_seed_sweep(
    lambda cw: DecisionTreeClassifier(
        **grid_dt.best_params_,
        class_weight=cw, random_state=42,
    ),
    df,
    n_seeds=50,
    model_name="DT",
    save_csv="sweep_dtree.csv",
    save_pickle="sweep_dtree.pkl",   # for re-plotting without re-computing
)

## Modeling
### DIAGNOSTIC PLOT 1: Feature Importances

In [ ]:
plot_sweep_feature_importances(
    sweep_dt,
    subtitle=f"hyperparams: max_depth={grid_dt.best_params_['max_depth']}, "
             f"criterion={grid_dt.best_params_['criterion']}",
)

### DIAGNOSTIC PLOT 2: Depth vs CV F1 Macro

In [ ]:
results_df = pd.DataFrame(grid_dt.cv_results_)
depth_labels = [str(d) if d is not None else "None" for d in [3, 5, 7, 10, None]]
depth_vals   = [3, 5, 7, 10, None]

fig, ax = plt.subplots(figsize=(10, 5))
for criterion, ls in [("gini", "-"), ("entropy", "--")]:
    means, stds = [], []
    for d in depth_vals:
        mask = (
            (results_df["param_criterion"] == criterion) &
            (results_df["param_max_depth"].apply(lambda x: x == d))
        )
        sub = results_df[mask]
        means.append(sub["mean_test_score"].max())
        stds.append(sub.loc[sub["mean_test_score"].idxmax(), "std_test_score"])
    means, stds = np.array(means), np.array(stds)
    ax.plot(range(len(depth_labels)), means, marker="o", ls=ls, label=f"criterion={criterion}")
    ax.fill_between(range(len(depth_labels)), means - stds, means + stds, alpha=0.15)

best_d = str(grid_dt.best_params_["max_depth"])
best_x = depth_labels.index(best_d)
ax.axvline(best_x, color="gray", ls=":", lw=1.5, label=f"Best max_depth={best_d}")
ax.set_xticks(range(len(depth_labels)))
ax.set_xticklabels(depth_labels, fontsize=10)
ax.set_xlabel("max_depth (None = unlimited)", fontsize=11)
ax.set_ylabel("CV F1 Macro (best over min_samples_leaf)", fontsize=11)
ax.set_title("Decision Tree — Depth vs CV F1 Macro", fontsize=13, fontweight="bold")
ax.legend(fontsize=10)
plt.tight_layout()
save_plot("dt_depth_sensitivity")
plt.show()

print("↑ Each point = best score across all min_samples_leaf values at that depth.")
print("  Shallow trees underfit; deep trees overfit the balanced training set.")
print("  None = no depth limit — the tree grows until leaves are pure.")

### DIAGNOSTIC PLOT 3: Tree Structure (depth ≤ 5)

In [ ]:
fig, ax = plt.subplots(figsize=(20, 8))
plot_tree(
    dt,
    max_depth=5,
    feature_names=FEATURES,
    class_names=class_names,
    filled=True,
    impurity=True,
    rounded=True,
    fontsize=8,
    ax=ax,
)
ax.set_title(
    f"Decision Tree — First 3 Levels (full depth = {dt.get_depth()}, leaves = {dt.get_n_leaves()})",
    fontsize=13, fontweight="bold",
)
plt.tight_layout()
save_plot("dt_tree_structure")
plt.show()

print("↑ Each node shows: split feature/threshold, Gini impurity, sample count, class distribution.")
print("  Colour intensity = class purity (darker = more confident split).")
print(f"  Only the first 3 levels shown — the full tree has {dt.get_depth()} levels and {dt.get_n_leaves()} leaves.")

## Metrics
### Classification Report + F1 Scores

In [ ]:
# Per-class F1 with mean ± std from the seed sweep
print_sweep_classification_report(sweep_dt)

# 5-fold CV on the balanced training set — orthogonal to sweep variance,
# measures within-seed tuning stability (different question).
cv_scores_dt = run_stratified_cv(dt, X_train_bal, y_train_bal)

# Cross-model comparison metrics: per-class P/R + aggregate robustness scores
# (MCC, balanced accuracy, minority F1 macro)
print_sweep_performance_summary(sweep_dt)

# Per-class bar chart with error bars from the sweep distribution
plot_sweep_f1_bar(sweep_dt)

### Confusion Matrices (Counts + Normalized)

In [ ]:
# Average row-normalized CM across all 50 seeds, annotated per-cell with std
plot_sweep_confusion_matrix(sweep_dt)

# Off-diagonal patterns: per-split count and per-class rate, both mean ± std
print_sweep_error_patterns(sweep_dt)

## Interpretation: